In [1]:
# Use google/gemma-4-12B-it
# Video-Based Physics Problem Evaluation

from google.colab import userdata
from huggingface_hub import login
import json
import time
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

In [2]:
## check if cuda is available and GPU

print("CUDA available:", torch.cuda.is_available())
print("Using GPU:", torch.cuda.get_device_name(0))

CUDA available: True
Using GPU: NVIDIA A100-SXM4-40GB


In [3]:
## authenticate with huggingface

hf_token = userdata.get("hf_token")
login(token=hf_token)

In [4]:
model_name ="gemma-4"
model_id = "google/gemma-4-12B-it"

# Load model
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

In [5]:
from pathlib import Path
import subprocess

# Clone the physics dataset from GitHub
repo_url = "https://github.com/ShRuwali/Physics_LLM_Test.git"
repo_dir = Path("/content/Physics_LLM_Test")

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", repo_url, str(repo_dir)],
        check=True
    )

# Locate the video files
video_dir = repo_dir / "Problems" / "Video"
video_files = sorted(video_dir.glob("*.mp4"))

# Create directories for model responses and metadata
output_dir = repo_dir / "Result" / "open_weight_models" / "video" / "gemma_4"
metadata_dir = output_dir / "metadata"

output_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

print("Total videos:", len(video_files))
print("Output directory:", output_dir)
print("Metadata directory:", metadata_dir)

Total videos: 24
Output directory: /content/Physics_LLM_Test/Result/open_weight_models/video/gemma_4
Metadata directory: /content/Physics_LLM_Test/Result/open_weight_models/video/gemma_4/metadata


In [6]:
import sys

# Import prompts from the video dataset
sys.path.insert(0, str(video_dir))

from video_prompts import regular_system_prompt, detail_system_prompt

# Check the number of prompts
print("Regular prompts:", len(regular_system_prompt))
print("Detailed prompts:", len(detail_system_prompt))
print("Total evaluations:", len(regular_system_prompt) + len(detail_system_prompt))

{'Problem_1_Regular_Q': 'You are a physics expert. From the video provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the video.', 'Problem_1_Obvious_Q': 'You are a physics expert. From the video provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the video.', 'Problem_1_non_Obvious_Q': 'You are a physics expert. From the video provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the video.', 'Problem_2_Regular_Q': 'You are a physics expert. From the video provide a brief description and calculate the time period of oscillation for small displacements. All the necessary values required are given in the video.', 'Problem_2_Obvious_Q': 'You are a physics expert. From the video provide a brief description and calculate th

In [7]:
# Match each video with its original prompt

video_lookup = {
    video.stem.lower(): video
    for video in video_files
}

prompt_groups = [
    ("regular_system_prompt", regular_system_prompt),
    ("detail_system_prompt", detail_system_prompt)
]

evaluations = []

for prompt_type, prompts in prompt_groups:
    for prompt_name, prompt_text in prompts.items():

        video_path = video_lookup.get(prompt_name.lower())

        if video_path is None:
            print("Missing video:", prompt_name)
            continue

        evaluations.append({
            "video_path": video_path,
            "prompt_name": prompt_name,
            "prompt_type": prompt_type,
            "prompt": prompt_text
        })

print("Total evaluations prepared:", len(evaluations))

print("\nRegular evaluations:",
      sum(e["prompt_type"] == "regular_system_prompt"
          for e in evaluations))

print("Detailed evaluations:",
      sum(e["prompt_type"] == "detail_system_prompt"
          for e in evaluations))

assert len(evaluations) == 40, "Expected 40 evaluations."

Total evaluations prepared: 40

Regular evaluations: 24
Detailed evaluations: 16


In [8]:
# Generate Gemma 4 responses for all video problems

model.eval()

for index, evaluation in enumerate(evaluations, start=1):

    video_path = evaluation["video_path"]
    prompt_name = evaluation["prompt_name"]
    prompt_type = evaluation["prompt_type"]
    system_prompt = evaluation["prompt"]

    result_name = f"{model_name}_{prompt_name}_{prompt_type}"

    result_file = output_dir / f"{result_name}.md"
    metadata_file = metadata_dir / f"{result_name}.json"

    # Skip evaluations that already have both result files
    if result_file.exists() and metadata_file.exists():
        print(f"[{index}/40] Already completed: {result_name}")
        continue

    print(f"[{index}/40] Processing: {result_name}")

    # Prepare the video and its corresponding prompt
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "video", "path": str(video_path)},
                {"type": "text", "text": system_prompt}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    input_length = inputs["input_ids"].shape[-1]

    # Measure model generation time
    torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10000,
            do_sample=False
        )

    torch.cuda.synchronize()
    total_time_seconds = time.perf_counter() - start_time

    # Decode the generated response
    generated_tokens = outputs[0, input_length:]

    output_tokens = generated_tokens.numel()

    response = processor.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # Save model response as Markdown
    result_file.write_text(
        response.strip(),
        encoding="utf-8"
    )

    # Save metadata using the same format as the image experiment
    metadata = {
        "requested_model": model_id,
        "problem_question": prompt_name,
        "prompt_name": system_prompt,
        "total_output_tokens": output_tokens,
        "total_time_taken": total_time_seconds
    }

    metadata_file.write_text(
        json.dumps(metadata, indent=4),
        encoding="utf-8"
    )

    print(
        f"Completed: {result_name} "
        f"({output_tokens} tokens, {total_time_seconds:.2f} seconds)"
    )

    # Release temporary tensors
    del inputs, outputs, generated_tokens

    torch.cuda.empty_cache()

print("\nExperiment finished!")
print("Markdown files:", len(list(output_dir.glob("*.md"))))
print("JSON metadata files:", len(list(metadata_dir.glob("*.json"))))

[1/40] Already completed: gemma-4_Problem_1_Regular_Q_regular_system_prompt
[2/40] Already completed: gemma-4_Problem_1_Obvious_Q_regular_system_prompt
[3/40] Already completed: gemma-4_Problem_1_non_Obvious_Q_regular_system_prompt
[4/40] Already completed: gemma-4_Problem_2_Regular_Q_regular_system_prompt
[5/40] Already completed: gemma-4_Problem_2_Obvious_Q_regular_system_prompt
[6/40] Already completed: gemma-4_Problem_2_non_obvious_Q_regular_system_prompt
[7/40] Already completed: gemma-4_Problem_3_Regular_Q_regular_system_prompt
[8/40] Already completed: gemma-4_Problem_3_Obvious_Q_regular_system_prompt
[9/40] Already completed: gemma-4_Problem_3_non_Obvious_Q_regular_system_prompt
[10/40] Already completed: gemma-4_Problem_4_Regular_Q_regular_system_prompt
[11/40] Already completed: gemma-4_Problem_4_Obvious_Q_regular_system_prompt
[12/40] Already completed: gemma-4_Problem_4_non_obvious_Q_regular_system_prompt
[13/40] Already completed: gemma-4_Problem_5_Regular_Q_regular_system